<a href="https://colab.research.google.com/github/kxenopoulou/xenopoulos-logic-dialectic/blob/main/%CE%B4%CE%B9%CE%B1%CE%BB%CE%B5%CE%BA%CF%84%CE%B9%CE%BA%CE%BF%CF%82_%CF%80%CE%BF%CE%BB%CE%B5%CE%BC%CE%BF%CF%82Untitled173.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# ΕΓΚΑΤΑΣΤΑΣΗ & ΡΥΘΜΙΣΕΙΣ
# ============================================
!pip install -q ipywidgets
!pip install -q seaborn

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Rescaling
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output
import seaborn as sns

warnings.filterwarnings('ignore')

# Ρυθμίσεις για Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except:
    pass

# Ρυθμίσεις οπτικοποίησης
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("✅ Βιβλιοθήκες φορτώθηκαν!")

# ============================================
# ΠΥΡΗΝΑΣ ΣΥΣΤΗΜΑΤΟΣ ΞΕΝΟΠΟΥΛΟΥ
# ============================================

class XenopoulosSystem:
    """Βελτιωμένη έκδοση του Γενετικο-Ιστορικού Συστήματος Λογικής"""

    def __init__(self, initial_state_A=0.3, historical_horizon=200,
                 aufhebung_threshold=0.85, system_name="Xenopoulos_LSTM_Analysis"):
        self.A = np.clip(initial_state_A, -1.5, 1.5)
        self.horizon = historical_horizon
        self.aufhebung_threshold = aufhebung_threshold
        self.system_name = system_name

        # Ιστορικά δεδομένα
        self.history_A = []
        self.history_anti_A = []
        self.history_tension = []
        self.history_XEPTQLRI = []
        self.history_stages = []
        self.history_stage_names = []
        self.risk_events = []
        self.paradox_events = []

        # Βελτιωμένοι ορισμοί σταδίων
        self.stages = {
            0: ("τ₀: Coherence", "#2E8B57", "✅"),
            1: ("τ₁: First Anomaly", "#3CB371", "⚠️"),
            2: ("τ₂: Anomaly Repetition", "#FFD700", "🔄"),
            3: ("τ₃: Meaning Incompatibility", "#FFA500", "⚡"),
            4: ("τ₄: System Saturation", "#FF6347", "🔥"),
            5: ("τ₅: Qualitative Leap (⤊)", "#DC143C", "⤊"),
            6: ("τ₆: Paradoxical Transcendence (⟡)", "#8A2BE2", "⟡"),
            7: ("τ₇: False Stability", "#FF69B4", "🎭"),
            8: ("τ₈: Permanent Dialectics", "#A9A9A9", "∞"),
            9: ("τ₉: Meta-Transcendence", "#000000", "🌀")
        }

        print(f"📊 Σύστημα Ξενόπουλου: '{system_name}'")
        print(f"   Αρχική κατάσταση: A = {self.A:.3f}")
        print(f"   Όρια ανίχνευσης: {len(self.stages)} στάδια")

    def _dialectical_negation(self, state):
        """Βελτιωμένος τελεστής ¬ᴰ με μνήμη και τυχαιότητα"""
        # Μνήμη από τα τελευταία 10 βήματα
        memory_effect = 0.0
        if len(self.history_A) > 0:
            window = min(10, len(self.history_A))
            recent_mean = np.mean(self.history_A[-window:])
            memory_effect = 0.2 * np.tanh(recent_mean * 2)

        # Παράγοντας διατήρησης (Xenopoulos: διατηρεί το Α)
        preservation = 0.7 + 0.3 * np.random.rand()

        # Βάρος ιστορικής τάσης
        historical_weight = 1.0 + 0.3 * np.random.rand()

        # Υπολογισμός άρνησης
        negation = -state * preservation * historical_weight * (1 + memory_effect)

        # Στοχαστική συνιστώσα με προσαρμοστικό πλάτος
        noise_level = 0.05 * (1 + abs(state))
        stochastic = noise_level * np.random.randn()

        return np.clip(negation + stochastic, -1.5, 1.5)

    def _calculate_tension(self, state, anti_state):
        """Διαλεκτική ένταση με μη-γραμμική κλιμάκωση"""
        raw_intensity = np.abs(state * anti_state)

        # Ενίσχυση για ακραίες τιμές
        if abs(state) > 0.8 and abs(anti_state) > 0.8:
            intensity = raw_intensity ** 0.7 * 1.5
        elif abs(state) > 0.6 or abs(anti_state) > 0.6:
            intensity = raw_intensity ** 0.8 * 1.2
        else:
            intensity = raw_intensity

        return np.clip(intensity, 0, 1)

    def _calculate_XEPTQLRI(self, tension, current_A, current_anti_A):
        """Βελτιωμένος υπολογισμός XEPTQLRI"""
        # 1. Βάση από ένταση (υπερβολική για μεγάλες τιμές)
        base_risk = tension ** 1.2

        # 2. Ιστορική τάση (τελευταία 5 βήματα)
        trend_factor = 1.0
        if len(self.history_tension) >= 5:
            recent_tension = self.history_tension[-5:]
            trend = np.polyfit(range(5), recent_tension, 1)[0]
            trend_factor = 1.0 + abs(trend) * 15

        # 3. Παράγοντας παραδόξου (ΚΡΙΤΙΚΟ)
        paradox_factor = 1.0
        if abs(current_A) > 0.8 and abs(current_anti_A) > 0.8:
            if tension < 0.35:  # Χαμηλή ένταση + ακραίες τιμές
                paradox_factor = 2.8
            else:
                paradox_factor = 2.0
        elif abs(current_A) > 0.9 or abs(current_anti_A) > 0.9:
            paradox_factor = 1.5

        # 4. Παράγοντας ασυμμετρίας
        asymmetry = abs(abs(current_A) - abs(current_anti_A))
        asymmetry_factor = 1.0 + (1 - asymmetry) * 0.5

        # 5. Τελικός υπολογισμός
        XEPTQLRI = (base_risk * trend_factor * paradox_factor * asymmetry_factor) / self.aufhebung_threshold

        # Τελική προσαρμογή με ομαλοποίηση
        final_XEPTQLRI = XEPTQLRI * (0.9 + 0.2 * np.random.rand())

        return np.clip(final_XEPTQLRI, 0, 3.5)

    def _classify_stage(self, tension, current_A, current_anti_A):
        """Βελτιωμένη ταξινόμηση με πολλαπλά κριτήρια"""
        stage_info = self.stages

        # ΚΡΙΤΗΡΙΟ 1: Παραδοξογενής Υπέρβαση (προτεραιότητα)
        if abs(current_A) > 0.85 and abs(current_anti_A) > 0.85:
            if tension < 0.4:
                return 6, stage_info[6]

        # ΚΡΙΤΗΡΙΟ 2: Ψευδής Σταθερότητα
        if tension < 0.25:
            if abs(current_A) > 0.75 or abs(current_anti_A) > 0.75:
                return 7, stage_info[7]

        # ΚΡΙΤΗΡΙΟ 3: Μόνιμη Διαλεκτική
        if len(self.history_stages) > 20:
            recent_stages = self.history_stages[-20:]
            stage_variability = np.std(recent_stages)
            if stage_variability > 1.8 and np.mean(recent_stages) > 3:
                return 8, stage_info[8]

        # ΚΡΙΤΗΡΙΟ 4: Κανονική ταξινόμηση βάσει έντασης
        if tension < 0.15:
            return 0, stage_info[0]
        elif tension < 0.30:
            return 1, stage_info[1]
        elif tension < 0.45:
            return 2, stage_info[2]
        elif tension < 0.60:
            return 3, stage_info[3]
        elif tension < self.aufhebung_threshold:
            return 4, stage_info[4]
        else:
            return 5, stage_info[5]

    def simulate_step(self, external_A_input):
        """Εκτέλεση ενός βήματος προσομοίωσης"""
        # Ενημέρωση κατάστασης Α
        self.A = np.clip(external_A_input, -1.5, 1.5)

        # 1. Διαλεκτική άρνηση
        current_anti_A = self._dialectical_negation(self.A)

        # 2. Διαλεκτική ένταση
        current_tension = self._calculate_tension(self.A, current_anti_A)

        # 3. Δείκτης XEPTQLRI
        current_XEPTQLRI = self._calculate_XEPTQLRI(current_tension, self.A, current_anti_A)

        # 4. Ταξινόμηση σε στάδιο
        stage_idx, (stage_name, stage_color, stage_icon) = self._classify_stage(
            current_tension, self.A, current_anti_A
        )

        # 5. Καταγραφή
        self.history_A.append(self.A)
        self.history_anti_A.append(current_anti_A)
        self.history_tension.append(current_tension)
        self.history_XEPTQLRI.append(current_XEPTQLRI)
        self.history_stages.append(stage_idx)
        self.history_stage_names.append(stage_name)

        # 6. Ανίχνευση γεγονότων
        if current_XEPTQLRI > 0.8:
            self.risk_events.append({
                'step': len(self.history_A) - 1,
                'XEPTQLRI': current_XEPTQLRI,
                'stage': stage_name,
                'A': self.A,
                'anti_A': current_anti_A,
                'tension': current_tension,
                'icon': '🔴'
            })

        if stage_idx == 6:  # Παράδοξο
            self.paradox_events.append({
                'step': len(self.history_A) - 1,
                'type': 'PARADOXICAL_TRANSCENDENCE',
                'A': self.A,
                'anti_A': current_anti_A,
                'tension': current_tension,
                'icon': '⟡'
            })

        return {
            'A': self.A,
            'anti_A': current_anti_A,
            'tension': current_tension,
            'XEPTQLRI': current_XEPTQLRI,
            'stage': stage_name,
            'stage_idx': stage_idx,
            'stage_color': stage_color,
            'stage_icon': stage_icon
        }

    def get_detailed_report(self):
        """Λεπτομερής αναλυτική έκθεση"""
        if not self.history_XEPTQLRI:
            return {"error": "Δεν υπάρχουν δεδομένα προσομοίωσης"}

        # Στατιστικές
        n_steps = len(self.history_A)
        mean_A = np.mean(self.history_A)
        mean_anti_A = np.mean(self.history_anti_A)
        mean_tension = np.mean(self.history_tension)
        mean_XEPTQLRI = np.mean(self.history_XEPTQLRI)
        max_XEPTQLRI = np.max(self.history_XEPTQLRI)

        # Χρόνος σε διαφορετικές καταστάσεις
        paradox_time = np.mean([1 if s == 6 else 0 for s in self.history_stages]) * 100
        false_stab_time = np.mean([1 if s == 7 else 0 for s in self.history_stages]) * 100
        critical_time = np.mean([1 if s in [5, 6, 7] else 0 for s in self.history_stages]) * 100

        # Συχνότητα σταδίων
        stage_counts = {}
        for idx, (name, color, icon) in self.stages.items():
            count = sum([1 for s in self.history_stages if s == idx])
            stage_counts[name] = count

        # Αξιολόγηση συνολικής κατάστασης
        if paradox_time > 40:
            status = "🔴 ΚΡΙΣΙΜΗ: Υπερβολική Παραδοξότητα"
            risk_level = "ΥΨΗΛΟΣ"
        elif false_stab_time > 50:
            status = "🟠 ΕΠΙΚΙΝΔΥΝΗ: Διερευνητική Ψευδής Σταθερότητα"
            risk_level = "ΜΕΣΟΣ-ΥΨΗΛΟΣ"
        elif max_XEPTQLRI > 2.0:
            status = "🟡 ΠΡΟΣΟΧΗ: Ενδείξεις Ακραίου Κινδύνου"
            risk_level = "ΜΕΣΟΣ"
        elif mean_XEPTQLRI < 0.4:
            status = "🟢 ΣΤΑΘΕΡΗ: Υγιής Διαλεκτική Δυναμική"
            risk_level = "ΧΑΜΗΛΟΣ"
        else:
            status = "🔵 ΔΥΝΑΜΙΚΗ: Ενεργή Διαλεκτική Εξέλιξη"
            risk_level = "ΜΕΣΟΣ"

        report = {
            'system_name': self.system_name,
            'total_steps': n_steps,
            'system_status': status,
            'risk_level': risk_level,
            'final_stage': self.history_stage_names[-1],
            'final_stage_icon': self.stages[self.history_stages[-1]][2],

            'statistics': {
                'mean_A': float(mean_A),
                'mean_anti_A': float(mean_anti_A),
                'mean_tension': float(mean_tension),
                'mean_XEPTQLRI': float(mean_XEPTQLRI),
                'max_XEPTQLRI': float(max_XEPTQLRI),
                'paradox_percentage': float(paradox_time),
                'false_stability_percentage': float(false_stab_time),
                'critical_states_percentage': float(critical_time)
            },

            'events': {
                'risk_events_count': len(self.risk_events),
                'paradox_events_count': len(self.paradox_events),
                'high_risk_events': len([e for e in self.risk_events if e['XEPTQLRI'] > 1.5])
            },

            'stage_distribution': stage_counts,

            'recommendations': self._generate_recommendations(
                paradox_time, false_stab_time, max_XEPTQLRI, mean_XEPTQLRI
            )
        }

        return report

    def _generate_recommendations(self, paradox_time, false_stab_time, max_XEPTQLRI, mean_XEPTQLRI):
        """Δημιουργία συμβουλών βάσει των αποτελεσμάτων"""
        recommendations = []

        if paradox_time > 30:
            recommendations.append("🎭 ΑΥΞΗΜΕΝΗ ΠΑΡΑΔΟΞΟΤΗΤΑ (>30%): Το σύστημα τείνει προς ταυτόχρονες ακραίες καταστάσεις. Εξετάστε διαφορετική αρχιτεκτονική ή δεδομένα εισόδου.")

        if false_stab_time > 40:
            recommendations.append("⚖️ ΨΕΥΔΗΣ ΣΤΑΘΕΡΟΤΗΤΑ (>40%): Η 'σταθερότητα' κρύβει αντιφάσεις. Προσθέστε stochastic components για να 'σπάσετε' την ψευδή ισορροπία.")

        if max_XEPTQLRI > 2.0:
            recommendations.append("⚠️ ΕΞΑΙΡΕΤΙΚΟΣ ΚΙΝΔΥΝΟΣ (XEPTQLRI>2.0): Κοντινό ποιοτικό άλμα. Παρακολουθήστε στενά και ετοιμαστείτε για αλλαγή παραμέτρων.")

        if mean_XEPTQLRI < 0.3 and paradox_time < 10:
            recommendations.append("✅ ΒΕΛΤΙΣΤΗ ΕΠΙΔΟΣΗ: Το σύστημα λειτουργεί σε υγιή διαλεκτική ισορροπία. Συνεχίστε την τρέχουσα προσέγγιση.")

        if len(recommendations) == 0:
            recommendations.append("📊 ΦΥΣΙΟΛΟΓΙΚΗ ΣΥΜΠΕΡΙΦΟΡΑ: Το σύστημα εξελίσσεται φυσιολογικά. Παρακολουθήστε περιοδικά για τυχόν αλλαγές.")

        return recommendations

print("✅ Πυρήνας Συστήματος Ξενόπουλου ολοκληρώθηκε!")

# ============================================
# ΒΕΛΤΙΩΜΕΝΟ LSTM ΜΟΝΤΕΛΟ
# ============================================

def generate_quantum_data(num_samples, timesteps=20, noise_level=0.5, seed=None):
    """Βελτιωμένη δημιουργία δεδομένων με έλεγχο θορύβου"""
    if seed is not None:
        np.random.seed(seed)

    # Καθαρές καταστάσεις με πιο ρεαλιστική δομή
    phase = np.random.uniform(0, 2*np.pi, (num_samples, 1, 1))
    amplitude = np.random.uniform(0.5, 1.5, (num_samples, timesteps, 3))

    clean_real = amplitude * np.cos(np.linspace(0, 4*np.pi, timesteps).reshape(1, -1, 1) + phase)
    clean_imag = amplitude * np.sin(np.linspace(0, 4*np.pi, timesteps).reshape(1, -1, 1) + phase)

    clean_states = clean_real + 1j * clean_imag
    clean_states = clean_states / (np.linalg.norm(clean_states, axis=-1, keepdims=True) + 1e-8)

    # Προσθετικός και πολλαπλασιαστικός θόρυβος
    additive_noise = noise_level * np.random.randn(*clean_states.shape)
    multiplicative_noise = 1 + (noise_level * 0.3) * np.random.randn(*clean_states.shape)

    noisy_states = clean_states * multiplicative_noise + additive_noise

    # Κανονικοποίηση
    noisy_states = noisy_states / (np.linalg.norm(noisy_states, axis=-1, keepdims=True) + 1e-8)

    # Μετατροπή σε πραγματικές/φανταστικές συνιστώσες
    noisy_states = np.float32(np.concatenate([noisy_states.real, noisy_states.imag], axis=-1))
    clean_states = np.float32(np.concatenate([clean_states.real, clean_states.imag], axis=-1))

    return noisy_states, clean_states

def generate_corrupted_quantum_data(num_samples, timesteps=20, noise_level=0.8, seed=None):
    """Δεδομένα με intentional contradictions και abrupt changes για ενδιαφέρουσα ανάλυση"""
    if seed is not None:
        np.random.seed(seed)

    # Κανονικά δεδομένα
    phase = np.random.uniform(0, 2*np.pi, (num_samples, 1, 1))
    amplitude = np.random.uniform(0.5, 1.5, (num_samples, timesteps, 3))

    clean_real = amplitude * np.cos(np.linspace(0, 4*np.pi, timesteps).reshape(1, -1, 1) + phase)
    clean_imag = amplitude * np.sin(np.linspace(0, 4*np.pi, timesteps).reshape(1, -1, 1) + phase)

    clean_states = clean_real + 1j * clean_imag
    clean_states = clean_states / (np.linalg.norm(clean_states, axis=-1, keepdims=True) + 1e-8)

    # Προσθετικός και πολλαπλασιαστικός θόρυβος
    additive_noise = noise_level * np.random.randn(*clean_states.shape)
    multiplicative_noise = 1 + (noise_level * 0.3) * np.random.randn(*clean_states.shape)

    noisy_states = clean_states * multiplicative_noise + additive_noise

    # Εισαγωγή διαφθοράς σε 40% των δειγμάτων
    corruption_mask = np.random.rand(num_samples) < 0.4

    for i in range(num_samples):
        if corruption_mask[i]:
            # Τύποι διαφθοράς (παραδόξου και αντιφάσεων)
            corruption_type = np.random.choice(['abrupt', 'paradox', 'contradiction', 'chaos'],
                                             p=[0.3, 0.3, 0.2, 0.2])

            if corruption_type == 'abrupt':
                # Abrupt change στο μέσο (διαστρωμάτωση)
                change_point = np.random.randint(8, 12)
                noisy_states[i, change_point:] *= -np.random.uniform(0.5, 2.0)  # Αντίθετη φάση με τυχαίο μέγεθος

            elif corruption_type == 'paradox':
                # Παραδοξότητα: ίδιες τιμές με αντίθετα labels
                paradox_segment = np.random.randint(0, timesteps-5)
                segment_length = np.random.randint(3, 7)
                # Δημιουργία παραδόξου: πολύ παρόμοιες τιμές αλλά με διαφορετική φάση
                paradox_factor = np.random.uniform(-0.3, 0.3)
                noisy_states[i, paradox_segment:paradox_segment+segment_length] *= (1 + paradox_factor)

            elif corruption_type == 'contradiction':
                # Αντίφαση: τοπικά ακραία σήματα
                contradiction_points = np.random.choice(timesteps, size=np.random.randint(2, 5), replace=False)
                for point in contradiction_points:
                    # Εκρήξεις τιμών
                    explosion_strength = np.random.uniform(1.5, 3.0)
                    noisy_states[i, point] *= explosion_strength

            elif corruption_type == 'chaos':
                # Χαοτική συμπεριφορά
                chaos_segment = np.random.randint(0, timesteps-8)
                segment_length = np.random.randint(6, 10)
                # Προσθήκη χάους (τυχαίες αλλαγές πρόσημου)
                chaos_pattern = np.random.choice([-1, 1], size=segment_length, p=[0.5, 0.5])
                noisy_states[i, chaos_segment:chaos_segment+segment_length] *= chaos_pattern.reshape(-1, 1)

    # Τελική κανονικοποίηση
    noisy_states = noisy_states / (np.linalg.norm(noisy_states, axis=-1, keepdims=True) + 1e-8)

    # Μετατροπή σε πραγματικές/φανταστικές συνιστώσες
    noisy_states = np.float32(np.concatenate([noisy_states.real, noisy_states.imag], axis=-1))
    clean_states = np.float32(np.concatenate([clean_states.real, clean_states.imag], axis=-1))

    return noisy_states, clean_states

def build_advanced_lstm_model(timesteps=20, input_dim=6, lstm_units=[256, 128, 64], dropout_rate=0.45):
    """Βελτιωμένο LSTM μοντέλο με παραμετροποίηση για πολυπλοκότητα"""
    input_layer = Input(shape=(timesteps, input_dim))

    # Προ-επεξεργασία με καλύτερη κλιμάκωση
    x = Rescaling(1./np.sqrt(input_dim))(input_layer)

    # Δυναμική αρχιτεκτονική με πολλά στρώματα
    for i, units in enumerate(lstm_units):
        return_sequences = (i < len(lstm_units) - 1)
        x = LSTM(units, return_sequences=return_sequences,
                activation='tanh', recurrent_activation='sigmoid',
                kernel_initializer='glorot_uniform',
                recurrent_dropout=dropout_rate*0.5)(x)

        if i < len(lstm_units) - 1:  # Dropout μόνο σε ενδιάμεσα στρώματα
            x = Dropout(dropout_rate)(x)

    # Επανάληψη για sequence output
    if not lstm_units[-1] == timesteps:
        x = tf.keras.layers.RepeatVector(timesteps)(x)
        x = LSTM(64, return_sequences=True, recurrent_dropout=dropout_rate*0.3)(x)

    # Βελτιωμένο output layer
    x = Dense(128, activation='relu')(x)
    x = Dropout(dropout_rate * 0.7)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout_rate * 0.5)(x)
    x = Dense(32, activation='relu')(x)
    output = Dense(input_dim, activation='linear')(x)

    model = Model(inputs=input_layer, outputs=output)
    return model

print("✅ LSTM μοντέλο ορίστηκε!")

# ============================================
# ΔΙΑΔΡΑΣΤΙΚΟ ΠΑΡΑΘΥΡΟ ΕΛΕΓΧΟΥ
# ============================================

class InteractiveXenopoulosAnalyzer:
    """Πλήρως διαδραστική ανάλυση LSTM με Ξενόπουλο"""

    def __init__(self):
        self.timesteps = 20
        self.input_dim = 6
        self.model = None
        self.xenopoulos_systems = []
        self.analysis_results = {}
        self.setup_widgets()

    def setup_widgets(self):
        """Δημιουργία διαδραστικών widgets"""
        print("🛠️  Δημιουργία διαδραστικού πίνακα ελέγχου...")

        # WIDGETS ΠΑΡΑΜΕΤΡΩΝ
        self.num_samples_slider = widgets.IntSlider(
            value=3000, min=1000, max=5000, step=500,
            description='Δείγματα:', style={'description_width': 'initial'}
        )

        self.noise_level_slider = widgets.FloatSlider(
            value=0.8, min=0.3, max=1.2, step=0.1,
            description='Θόρυβος:', style={'description_width': 'initial'}
        )

        self.epochs_slider = widgets.IntSlider(
            value=35, min=20, max=60, step=5,
            description='Epochs:', style={'description_width': 'initial'}
        )

        self.lstm_units_dropdown = widgets.Dropdown(
            options=['[128, 64]', '[256, 128, 64]', '[512, 256, 128]', '[512, 256, 128, 64]'],
            value='[256, 128, 64]',
            description='LSTM Units:', style={'description_width': 'initial'}
        )

        self.dropout_slider = widgets.FloatSlider(
            value=0.45, min=0.2, max=0.7, step=0.05,
            description='Dropout:', style={'description_width': 'initial'}
        )

        self.xen_steps_slider = widgets.IntSlider(
            value=150, min=50, max=300, step=25,
            description='Βήματα Ξενόπουλου:', style={'description_width': 'initial'}
        )

        self.data_corruption_dropdown = widgets.Dropdown(
            options=['Κανονικά', 'Με Διαφθορά', 'Εκρηκτικά'],
            value='Με Διαφθορά',
            description='Τύπος Δεδομένων:', style={'description_width': 'initial'}
        )

        # ΚΟΥΜΠΙΑ ΕΛΕΓΧΟΥ
        self.train_button = widgets.Button(
            description='🔥 ΕΚΚΙΝΗΣΗ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ',
            button_style='danger',
            tooltip='Εκτέλεση πλήρους ανάλυσης με extreme parameters',
            layout=widgets.Layout(width='350px', height='45px')
        )

        self.visualize_button = widgets.Button(
            description='📊 Οπτικοποίηση Αποτελεσμάτων',
            button_style='info',
            tooltip='Εμφάνιση γραφημάτων',
            disabled=True,
            layout=widgets.Layout(width='350px', height='45px')
        )

        # OUTPUT AREA
        self.output_area = widgets.Output(layout={'border': '1px solid #ccc', 'padding': '10px'})

        # ΣΥΝΔΕΣΗ ΚΟΥΜΠΙΩΝ
        self.train_button.on_click(self.run_full_analysis)
        self.visualize_button.on_click(self.visualize_results)

        # ΟΡΓΑΝΩΣΗ ΠΛΑΙΣΙΟΥ
        params_box = widgets.VBox([
            widgets.HTML("<h3>🔥 Παράμετροι Διαλεκτικού Πολέμου</h3>"),
            self.num_samples_slider,
            self.noise_level_slider,
            self.epochs_slider,
            self.lstm_units_dropdown,
            self.dropout_slider,
            self.xen_steps_slider,
            self.data_corruption_dropdown,
            widgets.HTML("<br>")
        ], layout=widgets.Layout(width='50%'))

        control_box = widgets.VBox([
            widgets.HTML("<h3>🎮 Έλεγχος</h3>"),
            self.train_button,
            widgets.HTML("<br>"),
            self.visualize_button
        ], layout=widgets.Layout(width='35%'))

        main_box = widgets.HBox([params_box, control_box])

        display(widgets.VBox([
            widgets.HTML("<h1 style='color: #ff6b6b;'>⚔️ ΔΙΑΛΕΚΤΙΚΟΣ ΠΟΛΕΜΟΣ: LSTM vs ΞΕΝΟΠΟΥΛΟΣ</h1>"),
            widgets.HTML("<p style='font-size: 14px; color: #666;'>Extreme παράμετροι για ενδιαφέρουσα ανάλυση με κίνδυνους, παραδόξα και αντιφάσεις</p>"),
            widgets.HTML("<div style='background: linear-gradient(to right, #ff6b6b, #ffa726); padding: 10px; border-radius: 5px;'>"
                        "<p style='color: white; margin: 0;'>⚠️  ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Αυτή η ανάλυση θα διαρκέσει 4-7 λεπτά αλλά θα αξίζει τον κόπο!</p>"
                        "</div>"),
            widgets.HTML("<br>"),
            main_box,
            self.output_area
        ]))

        print("✅ Διαδραστικός πίνακας ελέγχου έτοιμος!")

    def run_full_analysis(self, b):
        """Εκτέλεση πλήρους ανάλυσης"""
        with self.output_area:
            clear_output()
            print("="*70)
            print("🔥🔥🔥 ΕΚΚΙΝΗΣΗ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ ΞΕΝΟΠΟΥΛΟΥ 🔥🔥🔥")
            print("="*70)

            print("\n⚡ ΠΑΡΑΜΕΤΡΟΙ ΠΟΛΕΜΟΥ:")
            print(f"   • Δείγματα: {self.num_samples_slider.value}")
            print(f"   • Θόρυβος: {self.noise_level_slider.value} (EXTREME!)")
            print(f"   • Epochs: {self.epochs_slider.value}")
            print(f"   • LSTM Units: {self.lstm_units_dropdown.value}")
            print(f"   • Dropout: {self.dropout_slider.value}")
            print(f"   • Βήματα Ξενόπουλου: {self.xen_steps_slider.value}")
            print(f"   • Δεδομένα: {self.data_corruption_dropdown.value}")

            # 1. ΔΗΜΙΟΥΡΓΙΑ ΔΕΔΟΜΕΝΩΝ
            print("\n📁 ΒΗΜΑ 1: Δημιουργία δεδομένων...")

            if self.data_corruption_dropdown.value == 'Κανονικά':
                X, y = generate_quantum_data(
                    self.num_samples_slider.value,
                    self.timesteps,
                    self.noise_level_slider.value,
                    seed=42
                )
            else:  # Με Διαφθορά ή Εκρηκτικά
                X, y = generate_corrupted_quantum_data(
                    self.num_samples_slider.value,
                    self.timesteps,
                    self.noise_level_slider.value,
                    seed=42
                )
                if self.data_corruption_dropdown.value == 'Εκρηκτικά':
                    print("   💥 Εφαρμογή επιπλέον εκρηκτικών διαταραχών...")
                    # Επιπλέον διαταραχές
                    for i in range(X.shape[0]):
                        if np.random.rand() < 0.2:  # 20% των δειγμάτων
                            # Εκρηκτική διαταραχή
                            explosion_point = np.random.randint(0, self.timesteps-3)
                            explosion_strength = np.random.uniform(2.0, 4.0)
                            X[i, explosion_point:explosion_point+3] *= explosion_strength

            # Split
            split = int(0.8 * len(X))
            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]

            print(f"   • Training samples: {len(X_train)}")
            print(f"   • Test samples: {len(X_test)}")
            print(f"   • Shape: {X_train.shape}")

            # 2. ΚΑΤΑΣΚΕΥΗ ΜΟΝΤΕΛΟΥ
            print("\n🏗️  ΒΗΜΑ 2: Κατασκευή LSTM μοντέλου...")
            lstm_units = eval(self.lstm_units_dropdown.value)

            self.model = build_advanced_lstm_model(
                timesteps=self.timesteps,
                input_dim=self.input_dim,
                lstm_units=lstm_units,
                dropout_rate=self.dropout_slider.value
            )

            self.model.compile(
                optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
                loss='mse',
                metrics=['mae', 'mse']
            )

            print("   ✅ Μοντέλο κατασκευάστηκε!")

            # 3. ΕΚΠΑΙΔΕΥΣΗ
            print("\n📚 ΒΗΜΑ 3: Εκπαίδευση μοντέλου...")
            history = self.model.fit(
                X_train, y_train,
                epochs=self.epochs_slider.value,
                batch_size=64,
                validation_split=0.2,
                callbacks=[EarlyStopping(patience=7, restore_best_weights=True)],
                verbose=1
            )

            # 4. ΑΞΙΟΛΟΓΗΣΗ
            print("\n📊 ΒΗΜΑ 4: Αξιολόγηση απόδοσης...")
            test_results = self.model.evaluate(X_test, y_test, verbose=0)
            test_mae = test_results[1]
            test_mse = test_results[2]

            # Baseline
            baseline_mae = np.mean(np.abs(X_test - y_test))
            improvement = baseline_mae - test_mae
            improvement_pct = (improvement / baseline_mae) * 100

            print(f"   • Test MAE: {test_mae:.4f}")
            print(f"   • Test MSE: {test_mse:.4f}")
            print(f"   • Baseline MAE: {baseline_mae:.4f}")
            print(f"   • Βελτίωση: {improvement:.4f} ({improvement_pct:.1f}%)")

            # 5. ΔΙΑΛΕΚΤΙΚΗ ΑΝΑΛΥΣΗ ΜΕ ΞΕΝΟΠΟΥΛΟ
            print("\n🔮 ΒΗΜΑ 5: Διαλεκτική ανάλυση με Ξενόπουλο...")

            # Πρόβλεψη
            predictions = self.model.predict(X_test, verbose=0)

            # Υπολογισμός MAE ανά timestep
            mae_per_sample_timestep = np.abs(predictions - y_test)
            mae_per_timestep = np.mean(mae_per_sample_timestep, axis=(0, 2))

            # Αρχικοποίηση συστημάτων Ξενόπουλου
            self.xenopoulos_systems = []
            for step in range(self.timesteps):
                # Μετατροπή MAE σε κατάσταση Α (0=τέλειο, 1=χειρότερο)
                A_value = np.clip(1.0 - (mae_per_timestep[step] * 6), -1.0, 1.0)  # Χαλαρός πολλαπλασιαστής

                system = XenopoulosSystem(
                    initial_state_A=A_value,
                    historical_horizon=self.xen_steps_slider.value,
                    aufhebung_threshold=0.8,
                    system_name=f"LSTM_Step_{step}_MAE{mae_per_timestep[step]:.3f}"
                )

                # Προσομοίωση
                for i in range(self.xen_steps_slider.value):
                    # Προσθήκη τυχαιότητας που εξαρτάται από το MAE
                    noise_level = 0.08 + mae_per_timestep[step] * 0.15  # Περισσότερος θόρυβος
                    A_with_noise = A_value + np.random.normal(0, noise_level)
                    system.simulate_step(A_with_noise)

                self.xenopoulos_systems.append(system)

            # 6. ΣΥΓΚΕΝΤΡΩΤΙΚΗ ΑΝΑΛΥΣΗ
            print("\n📋 ΒΗΜΑ 6: Συγκεντρωτική ανάλυση αποτελεσμάτων...")

            all_reports = []
            high_risk_steps = []
            paradox_steps = []
            false_stability_steps = []

            for step, system in enumerate(self.xenopoulos_systems):
                report = system.get_detailed_report()
                all_reports.append(report)

                if report['statistics']['max_XEPTQLRI'] > 0.8:  # Χαμηλότερο όριο για περισσότερες εντοπίσεις
                    high_risk_steps.append((step, report['statistics']['max_XEPTQLRI']))

                if report['statistics']['paradox_percentage'] > 15.0:  # Χαμηλότερο όριο
                    paradox_steps.append((step, report['statistics']['paradox_percentage']))

                if report['statistics']['false_stability_percentage'] > 20.0:  # Χαμηλότερο όριο
                    false_stability_steps.append((step, report['statistics']['false_stability_percentage']))

            # Υπολογισμός μέσων όρων
            mean_XEPTQLRI = np.mean([r['statistics']['mean_XEPTQLRI'] for r in all_reports])
            mean_paradox = np.mean([r['statistics']['paradox_percentage'] for r in all_reports])
            mean_false_stab = np.mean([r['statistics']['false_stability_percentage'] for r in all_reports])

            # Αποθήκευση αποτελεσμάτων
            self.analysis_results = {
                'test_mae': test_mae,
                'test_mse': test_mse,
                'baseline_mae': baseline_mae,
                'improvement': improvement,
                'improvement_pct': improvement_pct,
                'mae_per_timestep': mae_per_timestep,
                'all_reports': all_reports,
                'high_risk_steps': high_risk_steps,
                'paradox_steps': paradox_steps,
                'false_stability_steps': false_stability_steps,
                'mean_XEPTQLRI': mean_XEPTQLRI,
                'mean_paradox': mean_paradox,
                'mean_false_stab': mean_false_stab,
                'model_history': history.history,
                'predictions': predictions,
                'X_test': X_test,
                'y_test': y_test
            }

            # 7. ΕΜΦΑΝΙΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ
            print("\n" + "="*70)
            print("📈 ΑΠΟΤΕΛΕΣΜΑΤΑ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ")
            print("="*70)

            print(f"\n📊 ΣΤΑΤΙΣΤΙΚΑ ΜΟΝΤΕΛΟΥ:")
            print(f"   • Test MAE: {test_mae:.4f}")
            print(f"   • Βελτίωση vs Baseline: {improvement_pct:.1f}%")

            print(f"\n⚠️  ΑΝΙΧΝΕΥΣΗ ΚΙΝΔΥΝΟΥ ΞΕΝΟΠΟΥΛΟΥ:")
            print(f"   • Μέσος XEPTQLRI: {mean_XEPTQLRI:.3f}")
            print(f"   • Μέση Παραδοξότητα: {mean_paradox:.1f}%")
            print(f"   • Μέση Ψευδής Σταθ.: {mean_false_stab:.1f}%")
            print(f"   • Βήματα με κίνδυνο: {len(high_risk_steps)}/{self.timesteps}")
            print(f"   • Βήματα με παράδοξο: {len(paradox_steps)}/{self.timesteps}")
            print(f"   • Βήματα με ψευδή σταθ.: {len(false_stability_steps)}/{self.timesteps}")

            if high_risk_steps:
                print(f"\n🔴 ΒΗΜΑΤΑ ΥΨΗΛΟΥ ΚΙΝΔΥΝΟΥ (XEPTQLRI > 0.8):")
                for step, risk in sorted(high_risk_steps, key=lambda x: x[1], reverse=True)[:5]:
                    print(f"   • Βήμα {step}: XEPTQLRI = {risk:.2f}, MAE = {mae_per_timestep[step]:.4f}")

            if paradox_steps:
                print(f"\n⟡ ΒΗΜΑΤΑ ΜΕ ΠΑΡΑΔΟΞΟΤΗΤΑ (>15%):")
                for step, paradox in sorted(paradox_steps, key=lambda x: x[1], reverse=True)[:5]:
                    print(f"   • Βήμα {step}: Παραδοξότητα = {paradox:.1f}%, MAE = {mae_per_timestep[step]:.4f}")

            if false_stability_steps:
                print(f"\n🎭 ΒΗΜΑΤΑ ΜΕ ΨΕΥΔΗ ΣΤΑΘΕΡΟΤΗΤΑ (>20%):")
                for step, false_stab in sorted(false_stability_steps, key=lambda x: x[1], reverse=True)[:5]:
                    print(f"   • Βήμα {step}: Ψευδής Σταθ. = {false_stab:.1f}%, MAE = {mae_per_timestep[step]:.4f}")

            # Αξιολόγηση συνολικής κατάστασης
            print(f"\n📌 ΣΥΝΟΛΙΚΗ ΔΙΑΓΝΩΣΗ:")
            if len(high_risk_steps) > self.timesteps / 2:
                print(f"   🔥 ΕΚΡΗΚΤΙΚΗ: Το μοντέλο βρίσκεται σε συνεχή κίνδυνο ποιοτικής αλλαγής!")
            elif len(paradox_steps) > self.timesteps / 3:
                print(f"   🎭 ΠΑΡΑΔΟΞΟΓΕΝΗΣ: Υψηλή παρουσία παραδόξων καταστάσεων")
            elif len(false_stability_steps) > self.timesteps / 4:
                print(f"   ⚖️ ΔΙΑΛΕΚΤΙΚΗ ΑΝΤΙΦΑΣΗ: Ενδείξεις ψευδούς σταθερότητας")
            elif improvement_pct > 30:
                print(f"   ⚔️ ΕΠΙΤΥΧΗΣ ΠΟΛΕΜΟΣ: Το μοντέλο νικά τον θόρυβο με διαλεκτική ισορροπία")
            else:
                print(f"   ⚡ ΔΥΝΑΜΙΚΗ ΜΑΧΗ: Ενεργή διαλεκτική εξέλιξη")

            # Ενεργοποίηση κουμπιού
            self.visualize_button.disabled = False

            print(f"\n✅ Η ΔΙΑΛΕΚΤΙΚΗ ΜΑΧΗ ΟΛΟΚΛΗΡΩΘΗΚΕ ΕΠΙΤΥΧΩΣ!")
            print(f"   Κάντε κλικ στο 'Οπτικοποίηση Αποτελεσμάτων' για τα γραφήματα.")

    def visualize_results(self, b):
        """Οπτικοποίηση αποτελεσμάτων"""
        with self.output_area:
            clear_output()
            print("🎨 ΔΗΜΙΟΥΡΓΙΑ ΟΠΤΙΚΟΠΟΙΗΣΕΩΝ...")

            if not self.analysis_results:
                print("⚠️  Δεν υπάρχουν αποτελέσματα. Εκτελέστε πρώτα την ανάλυση.")
                return

            # Δημιουργία συνοπτικού figure
            self.create_summary_figure()

            print("\n✅ Οπτικοποιήσεις δημιουργήθηκαν!")

    def create_summary_figure(self):
        """Δημιουργία συνοπτικού figure"""
        fig = plt.figure(figsize=(20, 16))

        # 1. MAE ανά timestep με ενδεικτικά σημεία
        ax1 = plt.subplot(3, 3, 1)
        mae = self.analysis_results['mae_per_timestep']
        timesteps = range(len(mae))

        bars = ax1.bar(timesteps, mae, alpha=0.7, edgecolor='black')

        # Χρώμα βάσει κινδύνου
        for i, (step, value) in enumerate(zip(timesteps, mae)):
            # Έλεγχος αν το βήμα είναι επικίνδυνο
            is_risky = any(step == risky_step for risky_step, _ in self.analysis_results['high_risk_steps'])
            is_paradox = any(step == paradox_step for paradox_step, _ in self.analysis_results['paradox_steps'])
            is_false_stab = any(step == false_step for false_step, _ in self.analysis_results['false_stability_steps'])

            if is_paradox:
                bars[i].set_color('#8A2BE2')  # Μωβ για παράδοξο
                bars[i].set_hatch('//')
                bars[i].set_alpha(0.9)
            elif is_risky:
                bars[i].set_color('#DC143C')  # Κόκκινο για κίνδυνο
                bars[i].set_hatch('\\')
                bars[i].set_alpha(0.9)
            elif is_false_stab:
                bars[i].set_color('#FF69B4')  # Ροζ για ψευδή σταθερότητα
                bars[i].set_hatch('xx')
                bars[i].set_alpha(0.9)
            elif value > np.mean(mae) * 1.3:
                bars[i].set_color('#FFA500')  # Πορτοκαλί για υψηλό MAE

        ax1.axhline(y=np.mean(mae), color='blue', linestyle='--',
                   label=f'Μέσος MAE: {np.mean(mae):.4f}')
        ax1.set_xlabel('Χρονικό Βήμα')
        ax1.set_ylabel('MAE')
        ax1.set_title('Απόδοση ανά Βήμα με Ένδειξη Κινδύνου')
        ax1.legend()
        ax1.grid(True, alpha=0.3, axis='y')

        # 2. Σύγκριση XEPTQLRI ανά βήμα
        ax2 = plt.subplot(3, 3, 2)
        max_XEPTQLRI = [r['statistics']['max_XEPTQLRI'] for r in self.analysis_results['all_reports']]
        mean_XEPTQLRI = [r['statistics']['mean_XEPTQLRI'] for r in self.analysis_results['all_reports']]

        ax2.plot(timesteps, max_XEPTQLRI, 'r-o', linewidth=2, markersize=6, label='Max XEPTQLRI')
        ax2.plot(timesteps, mean_XEPTQLRI, 'b-s', linewidth=2, markersize=4, alpha=0.7, label='Mean XEPTQLRI')

        ax2.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='Κίνδυνος (0.8)')
        ax2.axhline(y=1.0, color='darkred', linestyle='-', alpha=0.7, label='Υψηλός Κίνδυνος (1.0)')
        ax2.axhline(y=1.5, color='purple', linestyle=':', alpha=0.7, label='Εξαιρετικός (1.5)')

        # Σημειώσεις για κίνδυνο
        for step, risk in self.analysis_results['high_risk_steps']:
            if risk > 1.0:
                ax2.annotate(f'🔥 {risk:.1f}', xy=(step, risk), xytext=(step, risk+0.1),
                           arrowprops=dict(arrowstyle='->', color='red'), fontsize=9)

        ax2.set_xlabel('Χρονικό Βήμα')
        ax2.set_ylabel('XEPTQLRI')
        ax2.set_title('Δείκτης Κινδύνου Ξενόπουλου ανά Βήμα')
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)

        # 3. Παραδοξότητα vs Ψευδής Σταθερότητα
        ax3 = plt.subplot(3, 3, 3)

        paradox = [r['statistics']['paradox_percentage'] for r in self.analysis_results['all_reports']]
        false_stab = [r['statistics']['false_stability_percentage'] for r in self.analysis_results['all_reports']]

        width = 0.35
        x = np.arange(len(paradox))
        ax3.bar(x - width/2, paradox, width, label='Παραδοξότητα %', color='purple', alpha=0.7)
        ax3.bar(x + width/2, false_stab, width, label='Ψευδής Σταθ. %', color='orange', alpha=0.7)

        ax3.axhline(y=15, color='purple', linestyle=':', alpha=0.5, label='Όριο Παραδόξου')
        ax3.axhline(y=20, color='orange', linestyle=':', alpha=0.5, label='Όριο Ψευδούς Σταθ.')

        ax3.set_xlabel('Χρονικό Βήμα')
        ax3.set_ylabel('Ποσοστό (%)')
        ax3.set_title('Παραδοξότητα vs Ψευδής Σταθερότητα')
        ax3.legend(loc='upper right')
        ax3.grid(True, alpha=0.3, axis='y')

        # 4. Ιστορικό εκπαίδευσης
        ax4 = plt.subplot(3, 3, 4)

        history = self.analysis_results['model_history']
        epochs = range(1, len(history['loss']) + 1)

        ax4.plot(epochs, history['loss'], 'b-', label='Training Loss', linewidth=2)
        ax4.plot(epochs, history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Loss (MSE)')
        ax4.set_title('Ιστορικό Εκπαίδευσης')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        # 5. Heatmap των προβλέψεων vs πραγματικών τιμών
        ax5 = plt.subplot(3, 3, 5)

        if 'predictions' in self.analysis_results:
            sample_idx = np.random.randint(0, len(self.analysis_results['predictions']))
            predictions_sample = self.analysis_results['predictions'][sample_idx]
            actual_sample = self.analysis_results['y_test'][sample_idx]

            # Διαφορά
            diff = np.abs(predictions_sample - actual_sample)

            im = ax5.imshow(diff.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=np.max(diff))
            ax5.set_xlabel('Χρονικό Βήμα')
            ax5.set_ylabel('Διαστάσεις')
            ax5.set_title(f'Απόλυτη Διαφορά: Πρόβλεψη vs Πραγματικό (δείγμα {sample_idx})')
            plt.colorbar(im, ax=ax5, label='Απόλυτη Διαφορά')

            # Σημείωση για high-risk segments
            for step in range(diff.shape[0]):
                if np.mean(diff[step]) > np.mean(diff) * 1.5:
                    ax5.axvline(x=step, color='blue', linestyle='--', alpha=0.3, linewidth=1)

        # 6. Κατανομή σταδίων
        ax6 = plt.subplot(3, 3, 6)

        all_stages = []
        for system in self.xenopoulos_systems:
            all_stages.extend(system.history_stages)

        stage_counts = np.bincount(all_stages, minlength=10)
        stage_names = [f"τ{idx}" for idx in range(10)]

        colors = ['#2E8B57', '#3CB371', '#FFD700', '#FFA500', '#FF6347',
                 '#DC143C', '#8A2BE2', '#FF69B4', '#A9A9A9', '#000000']

        ax6.bar(range(10), stage_counts, color=colors, alpha=0.7, edgecolor='black')
        ax6.set_xlabel('Στάδιο')
        ax6.set_ylabel('Πλήθος')
        ax6.set_title('Κατανομή Διαλεκτικών Σταδίων')
        ax6.set_xticks(range(10))
        ax6.set_xticklabels(stage_names, rotation=45)
        ax6.grid(True, alpha=0.3, axis='y')

        # 7. Χάρτης θερμότητας A vs ¬A
        ax7 = plt.subplot(3, 3, 7)

        # Συλλογή όλων των τιμών A και anti-A
        all_A = []
        all_anti_A = []
        all_XEPTQLRI = []

        for system in self.xenopoulos_systems:
            all_A.extend(system.history_A)
            all_anti_A.extend(system.history_anti_A)
            all_XEPTQLRI.extend(system.history_XEPTQLRI)

        # Δειγματοληψία για απόδοση
        sample_size = min(500, len(all_A))
        indices = np.random.choice(len(all_A), sample_size, replace=False)

        scatter = ax7.scatter(np.array(all_A)[indices], np.array(all_anti_A)[indices],
                             c=np.array(all_XEPTQLRI)[indices], cmap='RdYlBu_r',
                             s=50, alpha=0.6, edgecolors='black', linewidth=0.3)

        # Ζώνες παραδόξου
        ax7.add_patch(plt.Rectangle((0.8, 0.8), 0.7, 0.7, alpha=0.15, color='red',
                               label='Ζώνη Παραδόξου I'))
        ax7.add_patch(plt.Rectangle((-1.5, -1.5), 0.7, 0.7, alpha=0.15, color='red',
                               label='Ζώνη Παραδόξου II'))

        ax7.set_xlabel('Τιμή A')
        ax7.set_ylabel('Τιμή ¬A')
        ax7.set_title('Φάσμα Καταστάσεων με XEPTQLRI')
        ax7.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax7.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax7.set_xlim(-1.5, 1.5)
        ax7.set_ylim(-1.5, 1.5)
        ax7.grid(True, alpha=0.2, linestyle='--')
        plt.colorbar(scatter, ax=ax7, label='XEPTQLRI')

        # 8. Χρονοσειρά τάσης
        ax8 = plt.subplot(3, 3, 8)

        # Επιλογή ενός ενδιαφέροντος συστήματος
        interesting_systems = []
        for idx, system in enumerate(self.xenopoulos_systems):
            report = self.analysis_results['all_reports'][idx]
            if (report['statistics']['max_XEPTQLRI'] > 0.8 or
                report['statistics']['paradox_percentage'] > 15):
                interesting_systems.append((idx, system))

        if interesting_systems:
            system_idx, system = interesting_systems[0]

            ax8.plot(system.history_A[:100], 'b-', label='A', linewidth=1.5)
            ax8.plot(system.history_anti_A[:100], 'r--', label='¬A', linewidth=1.5, alpha=0.7)
            ax8.fill_between(range(100), system.history_A[:100], system.history_anti_A[:100],
                           alpha=0.2, color='gray', label='Διαφορά')

            # Σημειώσεις για στάδια
            unique_stages = []
            for i in range(100):
                stage = system.history_stages[i]
                if i == 0 or stage != system.history_stages[i-1]:
                    unique_stages.append((i, stage))

            for i, stage in unique_stages[:5]:  # Πρώτες 5 αλλαγές
                color = system.stages[stage][1]
                ax8.axvline(x=i, color=color, linestyle=':', alpha=0.5, linewidth=1)
                ax8.annotate(f'τ{stage}', xy=(i, system.history_A[i]),
                           xytext=(i+2, system.history_A[i]), fontsize=8)

            ax8.set_xlabel('Βήμα Προσομοίωσης')
            ax8.set_ylabel('Τιμή Κατάστασης')
            ax8.set_title(f'Χρονοσειρά Συστήματος {system_idx} (A vs ¬A)')
            ax8.legend()
            ax8.grid(True, alpha=0.3)

        # 9. Αναλυτική εκθεση
        ax9 = plt.subplot(3, 3, 9)
        ax9.axis('off')

        results = self.analysis_results

        summary_text = (
            f"📋 ΑΝΑΛΥΤΙΚΗ ΕΚΘΕΣΗ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ\n"
            f"{'='*40}\n"
            f"• Test MAE: {results['test_mae']:.4f}\n"
            f"• Βελτίωση: {results['improvement_pct']:.1f}%\n"
            f"• Μέσος XEPTQLRI: {results['mean_XEPTQLRI']:.3f}\n\n"

            f"⚠️  ΕΝΔΕΙΞΕΙΣ ΚΙΝΔΥΝΟΥ:\n"
            f"• Βήματα κινδύνου: {len(results['high_risk_steps'])}/{self.timesteps}\n"
            f"• Βήματα παραδόξου: {len(results['paradox_steps'])}/{self.timesteps}\n"
            f"• Ψευδής σταθερότητα: {len(results['false_stability_steps'])}/{self.timesteps}\n\n"

            f"🎭 ΔΙΑΛΕΚΤΙΚΗ ΚΑΤΑΣΤΑΣΗ:\n"
        )

        if len(results['high_risk_steps']) > self.timesteps / 2:
            summary_text += "• 🔥 ΕΚΡΗΚΤΙΚΗ: Συνεχής κίνδυνος ποιοτικής αλλαγής\n"
        if len(results['paradox_steps']) > self.timesteps / 3:
            summary_text += "• 🎭 ΠΑΡΑΔΟΞΟΓΕΝΗΣ: Υψηλή παραδοξότητα\n"
        if len(results['false_stability_steps']) > self.timesteps / 4:
            summary_text += "• ⚖️ ΑΝΤΙΦΑΣΗ: Ενδείξεις ψευδούς σταθερότητας\n"

        if 'high_risk_steps' in results and results['high_risk_steps']:
            worst_step, worst_risk = max(results['high_risk_steps'], key=lambda x: x[1])
            summary_text += f"\n🔴 ΧΕΙΡΟΤΕΡΟ ΒΗΜΑ:\n"
            summary_text += f"• Βήμα {worst_step}: XEPTQLRI = {worst_risk:.2f}\n"
            summary_text += f"• MAE = {results['mae_per_timestep'][worst_step]:.4f}\n"

        ax9.text(0.05, 0.95, summary_text, transform=ax9.transAxes,
                fontsize=9, family='monospace', verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='#f8f9fa', alpha=0.9))

        plt.suptitle('📊 ΑΠΟΤΕΛΕΣΜΑΤΑ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ: LSTM vs ΣΥΣΤΗΜΑ ΞΕΝΟΠΟΥΛΟΥ\n',
                    fontsize=16, fontweight='bold', y=1.02, color='#ff6b6b')
        plt.tight_layout()
        plt.show()

print("✅ Το σύστημα διαλεκτικής ανάλυσης είναι έτοιμο!")

# ============================================
# ΕΚΚΙΝΗΣΗ ΤΟΥ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ
# ============================================

print("\n" + "="*70)
print("🔥🔥🔥 ΔΙΑΛΕΚΤΙΚΟΣ ΠΟΛΕΜΟΣ: LSTM vs ΣΥΣΤΗΜΑ ΞΕΝΟΠΟΥΛΟΥ 🔥🔥🔥")
print("="*70)
print("\n⚔️  ΠΑΡΑΜΕΤΡΟΙ ΠΟΛΕΜΟΥ:")
print("   • Θόρυβος: 0.8 (EXTREME!)")
print("   • Δείγματα: 3000")
print("   • Epochs: 35")
print("   • LSTM: [256, 128, 64]")
print("   • Dropout: 0.45")
print("   • Βήματα Ξενόπουλου: 150")
print("   • Δεδομένα: Με Διαφθορά")
print("\n🎯 ΑΝΑΜΕΝΟΜΕΝΑ ΑΠΟΤΕΛΕΣΜΑΤΑ:")
print("   • Περισσότερα 'paradox events' και κίνδυνοι")
print("   • Μεταβάσεις μεταξύ διαλεκτικών σταδίων")
print("   • XEPTQLRI > 1.0 σε πολλά βήματα")
print("   • Ενδιαφέρουσες αντιφάσεις και εκρήξεις")
print("\n⏱️  ΧΡΟΝΟΣ ΕΚΤΕΛΕΣΗΣ: 4-7 λεπτά")
print("\n🚀 ΚΑΝΕ ΚΛΙΚ ΣΤΟ '🔥 ΕΚΚΙΝΗΣΗ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ' ΓΙΑ ΝΑ ΞΕΚΙΝΗΣΕΙ!")

# Αρχικοποίηση του αναλυτή
analyzer = InteractiveXenopoulosAnalyzer()

# ============================================
# ΕΠΙΠΛΕΟΝ ΕΡΓΑΛΕΙΑ ΓΙΑ ΑΝΑΛΥΣΗ
# ============================================

def create_detailed_paradox_report():
    """Δημιουργία λεπτομερούς αναφοράς για παραδόξα και κινδύνους"""
    if not hasattr(analyzer, 'analysis_results') or not analyzer.analysis_results:
        print("⚠️  Δεν υπάρχουν αποτελέσματα. Εκτελέστε πρώτα την ανάλυση.")
        return

    results = analyzer.analysis_results

    print("\n" + "="*80)
    print("🔍 ΛΕΠΤΟΜΕΡΗΣ ΑΝΑΛΥΣΗ ΠΑΡΑΔΟΞΩΝ ΚΑΙ ΚΙΝΔΥΝΩΝ")
    print("="*80)

    # 1. Στατιστικά
    print("\n📊 ΣΤΑΤΙΣΤΙΚΑ ΜΟΝΤΕΛΟΥ:")
    print(f"   • Test MAE: {results['test_mae']:.4f}")
    print(f"   • Βελτίωση: {results['improvement_pct']:.1f}%")
    print(f"   • Μέσος XEPTQLRI: {results['mean_XEPTQLRI']:.3f}")

    # 2. Κίνδυνοι ανά βήμα
    print("\n⚠️  ΚΙΝΔΥΝΟΙ ΑΝΑ ΒΗΜΑ:")
    for step, mae in enumerate(results['mae_per_timestep']):
        is_risky = any(step == risky_step for risky_step, _ in results['high_risk_steps'])
        is_paradox = any(step == paradox_step for paradox_step, _ in results['paradox_steps'])
        is_false = any(step == false_step for false_step, _ in results['false_stability_steps'])

        indicators = []
        if is_risky: indicators.append("🔴")
        if is_paradox: indicators.append("⟡")
        if is_false: indicators.append("🎭")

        if indicators:
            print(f"   Βήμα {step:2d}: MAE={mae:.4f} {' '.join(indicators)}")

    # 3. Λεπτομέρειες για high-risk βήματα
    if results['high_risk_steps']:
        print("\n🔴 ΛΕΠΤΟΜΕΡΕΙΕΣ ΓΙΑ ΒΗΜΑΤΑ ΥΨΗΛΟΥ ΚΙΝΔΥΝΟΥ:")
        for step, risk in sorted(results['high_risk_steps'], key=lambda x: x[1], reverse=True):
            report = results['all_reports'][step]
            stats = report['statistics']
            print(f"\n   Βήμα {step}:")
            print(f"      • XEPTQLRI: {risk:.2f}")
            print(f"      • MAE: {results['mae_per_timestep'][step]:.4f}")
            print(f"      • Στάδιο: {report['final_stage_icon']} {report['final_stage']}")
            print(f"      • Παραδοξότητα: {stats['paradox_percentage']:.1f}%")
            print(f"      • Ψευδής Σταθ.: {stats['false_stability_percentage']:.1f}%")

    # 4. Συστήματα με ενδιαφέρουσες συμπεριφορές
    print("\n🎭 ΣΥΣΤΗΜΑΤΑ ΜΕ ΕΝΔΙΑΦΕΡΟΥΣΕΣ ΔΙΑΛΕΚΤΙΚΕΣ ΣΥΜΠΕΡΙΦΟΡΕΣ:")
    interesting_count = 0
    for step, system in enumerate(analyzer.xenopoulos_systems):
        report = results['all_reports'][step]
        stats = report['statistics']

        if (stats['max_XEPTQLRI'] > 1.0 or
            stats['paradox_percentage'] > 25 or
            stats['false_stability_percentage'] > 30):

            interesting_count += 1
            print(f"\n   Σύστημα {step} ({system.system_name}):")
            print(f"      • Κατάσταση: {report['system_status']}")
            print(f"      • Max XEPTQLRI: {stats['max_XEPTQLRI']:.2f}")
            print(f"      • Παραδοξότητα: {stats['paradox_percentage']:.1f}%")
            print(f"      • Ψευδής Σταθ.: {stats['false_stability_percentage']:.1f}%")

            # Εμφάνιση πρότεισης
            if stats['paradox_percentage'] > 30:
                print(f"      💡 ΠΡΟΤΑΣΗ: Υπερβολική παραδοξότητα - ελέγξτε overfitting")
            elif stats['false_stability_percentage'] > 40:
                print(f"      💡 ΠΡΟΤΑΣΗ: Ψευδής σταθερότητα - προσθέστε θόρυβο")

    if interesting_count == 0:
        print("   Δεν βρέθηκαν συστήματα με εξαιρετικά ενδιαφέρουσες συμπεριφορές.")

    # 5. Συνοπτική διάγνωση
    print("\n📌 ΣΥΝΟΛΙΚΗ ΔΙΑΓΝΩΣΗ:")
    total_risks = len(results['high_risk_steps']) + len(results['paradox_steps']) + len(results['false_stability_steps'])
    risk_percentage = (total_risks / (self.timesteps * 3)) * 100

    if risk_percentage > 50:
        print("   🔥 ΕΚΡΗΚΤΙΚΗ ΣΥΜΠΕΡΙΦΟΡΑ: Το μοντέλο λειτουργεί σε συνεχή κίνδυνο")
        print("   💡 ΠΡΟΤΑΣΗ: Μείωση πολυπλοκότητας ή αύξηση regularization")
    elif risk_percentage > 30:
        print("   ⚠️  ΔΥΝΑΜΙΚΗ ΣΥΜΠΕΡΙΦΟΡΑ: Σημαντική παρουσία διαλεκτικών αντιφάσεων")
        print("   💡 ΠΡΟΤΑΣΗ: Συνεχίστε με προσοχή - το μοντέλο είναι ενεργό")
    elif risk_percentage > 15:
        print("   🔵 ΜΕΤΡΙΑ ΣΥΜΠΕΡΙΦΟΡΑ: Μερικοί κίνδυνοι αλλά ελεγχόμενοι")
        print("   💡 ΠΡΟΤΑΣΗ: Βελτιστοποίηση παραμέτρων για καλύτερη απόδοση")
    else:
        print("   🟢 ΣΤΑΘΕΡΗ ΣΥΜΠΕΡΙΦΟΡΑ: Χαμηλός κίνδυνος, σταθερή απόδοση")
        print("   💡 ΠΡΟΤΑΣΗ: Συνεχίστε την τρέχουσα προσέγγιση")

    print(f"\n   Στατιστικά κινδύνου: {risk_percentage:.1f}%")
    print(f"   Συνολικοί ενδείξεις: {total_risks}")

# Προσθήκη της συνάρτησης στο global scope
import sys
module = sys.modules[__name__]
setattr(module, 'create_detailed_paradox_report', create_detailed_paradox_report)

print("\n✅ ΕΠΙΠΛΕΟΝ ΕΡΓΑΛΕΙΑ:")
print("   • Μπορείτε να καλέσετε τη συνάρτηση 'create_detailed_paradox_report()'")
print("     μετά την ολοκλήρωση της ανάλυσης για λεπτομερή αναφορά.")
print("\n🚀 ΟΛΟΚΛΗΡΩΜΕΝΟ ΣΥΣΤΗΜΑ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ ΕΤΟΙΜΟ!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.3 MB/s eta 0:00:00
✅ Βιβλιοθήκες φορτώθηκαν!
✅ Πυρήνας Συστήματος Ξενόπουλου ολοκληρώθηκε!
✅ LSTM μοντέλο ορίστηκε!
✅ Το σύστημα διαλεκτικής ανάλυσης είναι έτοιμο!

🔥🔥🔥 ΔΙΑΛΕΚΤΙΚΟΣ ΠΟΛΕΜΟΣ: LSTM vs ΣΥΣΤΗΜΑ ΞΕΝΟΠΟΥΛΟΥ 🔥🔥🔥

⚔️  ΠΑΡΑΜΕΤΡΟΙ ΠΟΛΕΜΟΥ:
   • Θόρυβος: 0.8 (EXTREME!)
   • Δείγματα: 3000
   • Epochs: 35
   • LSTM: [256, 128, 64]
   • Dropout: 0.45
   • Βήματα Ξενόπουλου: 150
   • Δεδομένα: Με Διαφθορά

🎯 ΑΝΑΜΕΝΟΜΕΝΑ ΑΠΟΤΕΛΕΣΜΑΤΑ:
   • Περισσότερα 'paradox events' και κίνδυνοι
   • Μεταβάσεις μεταξύ διαλεκτικών σταδίων
   • XEPTQLRI > 1.0 σε πολλά βήματα
   • Ενδιαφέρουσες αντιφάσεις και εκρήξεις

⏱️  ΧΡΟΝΟΣ ΕΚΤΕΛΕΣΗΣ: 4-7 λεπτά

🚀 ΚΑΝΕ ΚΛΙΚ ΣΤΟ '🔥 ΕΚΚΙΝΗΣΗ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ' ΓΙΑ ΝΑ ΞΕΚΙΝΗΣΕΙ!
🛠️  Δημιουργία διαδραστικού πίνακα ελέγχου...


✅ Διαδραστικός πίνακας ελέγχου έτοιμος!

✅ ΕΠΙΠΛΕΟΝ ΕΡΓΑΛΕΙΑ:
   • Μπορείτε να καλέσετε τη συνάρτηση 'create_detailed_paradox_report()'
     μετά την ολοκλήρωση της ανάλυσης για λεπτομερή αναφορά.

🚀 ΟΛΟΚΛΗΡΩΜΕΝΟ ΣΥΣΤΗΜΑ ΔΙΑΛΕΚΤΙΚΟΥ ΠΟΛΕΜΟΥ ΕΤΟΙΜΟ!


In [3]:
create_detailed_paradox_report()


🔍 ΛΕΠΤΟΜΕΡΗΣ ΑΝΑΛΥΣΗ ΠΑΡΑΔΟΞΩΝ ΚΑΙ ΚΙΝΔΥΝΩΝ

📊 ΣΤΑΤΙΣΤΙΚΑ ΜΟΝΤΕΛΟΥ:
   • Test MAE: 0.1266
   • Βελτίωση: 55.7%
   • Μέσος XEPTQLRI: 0.107

⚠️  ΚΙΝΔΥΝΟΙ ΑΝΑ ΒΗΜΑ:
   Βήμα  4: MAE=0.1206 🔴
   Βήμα  5: MAE=0.1224 🔴
   Βήμα  6: MAE=0.1247 🔴
   Βήμα  7: MAE=0.1287 🔴
   Βήμα 10: MAE=0.1251 🔴
   Βήμα 13: MAE=0.1195 🔴
   Βήμα 14: MAE=0.1236 🔴
   Βήμα 15: MAE=0.1227 🔴
   Βήμα 17: MAE=0.1275 🔴

🔴 ΛΕΠΤΟΜΕΡΕΙΕΣ ΓΙΑ ΒΗΜΑΤΑ ΥΨΗΛΟΥ ΚΙΝΔΥΝΟΥ:

   Βήμα 17:
      • XEPTQLRI: 1.53
      • MAE: 0.1275
      • Στάδιο: ✅ τ₀: Coherence
      • Παραδοξότητα: 0.0%
      • Ψευδής Σταθ.: 0.0%

   Βήμα 6:
      • XEPTQLRI: 1.23
      • MAE: 0.1247
      • Στάδιο: ⚠️ τ₁: First Anomaly
      • Παραδοξότητα: 0.0%
      • Ψευδής Σταθ.: 0.0%

   Βήμα 4:
      • XEPTQLRI: 1.21
      • MAE: 0.1206
      • Στάδιο: ✅ τ₀: Coherence
      • Παραδοξότητα: 0.0%
      • Ψευδής Σταθ.: 0.0%

   Βήμα 10:
      • XEPTQLRI: 0.98
      • MAE: 0.1251
      • Στάδιο: ✅ τ₀: Coherence
      • Παραδοξότητα: 0.0%
      • Ψευδής Σταθ.: 0.

NameError: name 'self' is not defined